In [13]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [14]:
gemini_api_key=os.getenv("GEMINI_API_KEY")
gemini_base_url=os.getenv("GEMINI_BASE_URL")

In [15]:
gemini=OpenAI(base_url=gemini_base_url, api_key=gemini_api_key)

In [23]:
def message_gpt(message):
    messages=[{'role':'system','content':'You are a helpful assistant.'},
              {'role':'user','content':message}]
    response=gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
    return response.choices[0].message.content 

In [24]:
message_gpt("What is today's date?")

'Today is Wednesday, May 22, 2024.'

# Gradio Time

In [31]:
#creating a temp function to test gradio interface
def greet(name):
    return f'Hello, {name} nice to meet you!'

In [ ]:

message_input = gr.Textbox(label="Your message to GPT", info="Enter a message for GPT to respond to", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT Chat Interface", 
    inputs=[message_input], 
    outputs=[message_output], 
    flagging_mode="never"
    )
view.launch(inbrowser=True)

# Allowing the user to select their own model 

In [45]:
#we give the model a system prompt to set the context for the conversation, and then we give it a user prompt which is the message we want it to respond to. The model then generates a response based on the system and user prompts.
system_prompt = "You are a helpful assistant."

Gemini Model:  gemini-3.1-flash

In [46]:
#1. GEMINI Model: gemini-3.1-flash-lite
gemini_api_key=os.getenv("GEMINI_API_KEY")
gemini_base_url=os.getenv("GEMINI_BASE_URL")
gemini=OpenAI(base_url=gemini_base_url, api_key=gemini_api_key)

def gemini_model(message):
    message=[{'role':'system','content':system_prompt},
             {'role':'user','content':message}]
    response=gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=message)
    return response.choices[0].message.content


Ollama Model: Llama3.2 

In [56]:
#2. Ollama Model: llama3.2 
ollama_api_key=os.getenv('OLLAMA_API_KEY')
ollama_base_url=os.getenv('OLLAMA_BASE_URL')
ollama=OpenAI(base_url=ollama_base_url, api_key=ollama_api_key)

def ollama_model(message):
    message=[{'role':'system','content':system_prompt},
             {'role':'user','content':message}]
    response=ollama.chat.completions.create(model="llama3.2", messages=message)
    return response.choices[0].message.content

In [57]:
#model selection function:
def models(message,model):
    if model=='gemini_model':
        return gemini_model(message)
    elif model=='ollama_model':
        return ollama_model(message)
    else:
        return ValueError("Invalid model name. Please choose either 'gemini_model' or 'ollama_model'.")

In [ ]:
#creating a gradio interface to test the model selection function
message_input=gr.Textbox(label="Your message to GPT", info="Enter a message for GPT to respond to", lines=7)
model_input=gr.Dropdown(label="Select a model", choices=['gemini_model', 'ollama_model'])
message_output=gr.Textbox(label="Response:", lines=8)

view=gr.Interface(fn=models,title="Model Selection Interface", inputs=[message_input, model_input], outputs=[message_output], flagging_mode="never").launch(inbrowser=True, auth=("hari", "hari123"))